# NYC Congestion Pricing — SubPop Full Pipeline (Colab)

**Runtime required:** GPU (L4 recommended, T4 works for zero-shot, A100 for fine-tuning)

In Colab: Runtime → Change runtime type → L4 GPU

---
**Order of cells:**
1. Mount Drive + clone repo
2. Install dependencies
3. HuggingFace login
4. Step 2 — zero-shot baselines (GPU)
5. Step 3 — prepare fine-tuning data (CPU)
6. Step 4 — LoRA fine-tuning (GPU)
7. Step 5 — fine-tuned inference (GPU)
8. Step 6 — evaluation (CPU)
9. Save results to Drive

## Cell 1 — Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Subpop_replication'
DRIVE_DIR = '/content/drive/MyDrive/Subpop_replication_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Clone or pull latest code
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Twyla123/Subpop_replication.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## Cell 2 — Install dependencies

In [ ]:
# Install GPU dependencies
!pip install -q vllm
!pip install -q peft accelerate bitsandbytes datasets fire tqdm
!pip install -q transformers>=4.40.0

# Install the SubPop library
!cd subpop-main && pip install -q -e . && cd ..

print('All dependencies installed.')

## Cell 3 — HuggingFace login

In [ ]:
# ── Load HuggingFace token from Colab Secrets ──────────────────────────────
# In Colab: click the 🔑 key icon in the left sidebar
#           → Add new secret → Name: HF_TOKEN → Value: hf_...
# This keeps your token out of the notebook and off GitHub.
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)
print('HuggingFace login successful.')

# Primary model — Llama-3.1-8B (access approved)
# For Mistral comparison, see optional Cell 10 at the bottom.
MODEL_NAME = 'meta-llama/Llama-3.1-8B'

## Cell 4 — Step 2: Zero-shot baselines (GPU, ~10 min per format)
Produces `distributions_QA.csv`, `distributions_BIO.csv`, `distributions_PORTRAY.csv`

In [ ]:
!python step2_vllm_baselines.py \
    --mode zeroshot \
    --model_name {MODEL_NAME} \
    --demographics_csv  approach2_outputs/cms/cms_demographics.csv \
    --steering_json     approach2_outputs/cms/cms_steering_prompts.json \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --output_dir        approach2_outputs/cms \
    --formats QA BIO PORTRAY

In [ ]:
# Statistical bounds — uses WD for ordinal questions, TVD for nominal questions
# --questions_json provides the is_ordinal flag per question
!python step2_vllm_baselines.py \
    --mode bounds \
    --ground_truth_csv  approach2_outputs/cms/cms_survey_distributions.csv \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --output_dir        approach2_outputs/cms

## Cell 5 — Step 3: Prepare fine-tuning data (CPU, ~1 min)
Produces `cms_QA_train.csv`, `cms_BIO_train.csv`, etc. in `finetuning_data/`

In [ ]:
for fmt in ['QA', 'BIO', 'PORTRAY', 'ALL']:
    pass  # generated together in one call below

!python step3_prepare_finetuning_data.py \
    --distributions_csv   approach2_outputs/cms/cms_survey_distributions.csv \
    --demographics_csv    approach2_outputs/cms/cms_demographics.csv \
    --steering_json       approach2_outputs/cms/cms_steering_prompts.json \
    --question_split_json approach2_outputs/cms/cms_question_split.json \
    --output_dir          approach2_outputs/cms/finetuning_data \
    --formats QA BIO PORTRAY ALL

## Cell 6 — Step 4: LoRA Fine-tuning (GPU, ~30 min on L4)

Run once per steering format. Start with QA (matches SubPop paper).

Checkpoints saved to `approach2_outputs/cms/checkpoints_QA/`

In [ ]:
import os

os.chdir('subpop-main')

exit_code = os.system(f"""python scripts/experiment/run_finetune.py \
    --model_name        {MODEL_NAME} \
    --dataset           cms_dataset \
    --steering_type     QA \
    --output_dir        ../approach2_outputs/cms/checkpoints_QA/ \
    --num_epochs        50 \
    --batch_size_training 4 \
    --lr                2e-4 \
    --use_peft \
    --quantization \
    --loss_function_type ce \
    --enable_fsdp=False \
    --use_wandb=False""")

os.chdir('..')

if exit_code == 0:
    print('Fine-tuning complete.')
else:
    raise RuntimeError(
        f'Fine-tuning FAILED (exit code {exit_code}). '
        'Check the output above for the error before continuing to Cell 7.'
    )

In [ ]:
import os
os.chdir('subpop-main')

exit_code = os.system(f"""python scripts/experiment/run_finetune.py \
    --model_name        {MODEL_NAME} \
    --dataset           cms_dataset \
    --steering_type     QA \
    --output_dir        ../approach2_outputs/cms/checkpoints_QA/ \
    --num_epochs        50 \
    --batch_size_training 4 \
    --lr                2e-4 \
    --use_peft \
    --quantization \
    --loss_function_type ce \
    --enable_fsdp=False""")

os.chdir('..')

if exit_code == 0:
    print('Fine-tuning complete.')
else:
    raise RuntimeError(f'Fine-tuning FAILED (exit code {exit_code}). Check output above.')

In [ ]:
import glob, os, pathlib

# ── Locate checkpoint ────────────────────────────────────────────────────────
# finetuning.py appended a timestamp to the trailing slash we passed, so the
# checkpoint lives at  approach2_outputs/cms/checkpoints_QA/<timestamp>/
checkpoints = sorted(glob.glob('approach2_outputs/cms/checkpoints_QA/*'))
if not checkpoints:
    raise FileNotFoundError(
        'No checkpoint found under approach2_outputs/cms/checkpoints_QA/\n'
        'Did Cell 6 (step 4) complete successfully?'
    )
checkpoint_path = checkpoints[-1]
print(f'Using checkpoint: {checkpoint_path}')

# ── Run inference ────────────────────────────────────────────────────────────
TEST_CSV   = 'approach2_outputs/cms/finetuning_data/cms_QA_test.csv'
LORA_NAME  = 'cms_QA'
INFER_DIR  = 'approach2_outputs/cms/'   # trailing slash required by run_inference.py

os.chdir('subpop-main')
!python scripts/experiment/run_inference.py \
    --input_paths             ../{TEST_CSV} \
    --output_dir              ../{INFER_DIR} \
    --base_model_name_or_path {MODEL_NAME} \
    --lora_path               ../{checkpoint_path} \
    --lora_name               {LORA_NAME}
os.chdir('..')

# ── Rename output to canonical name expected by step 6 ──────────────────────
# run_inference.py writes:  {output_dir}{input_stem}_{lora_name}.csv
raw_stem  = pathlib.Path(TEST_CSV).stem                       # cms_QA_test
raw_out   = f'{INFER_DIR}{raw_stem}_{LORA_NAME}.csv'          # .../cms_QA_test_cms_QA.csv
canonical = 'approach2_outputs/cms/results_finetuned_QA.csv'

if os.path.exists(raw_out):
    os.rename(raw_out, canonical)
    print(f'Renamed: {raw_out} → {canonical}')
else:
    print(f'WARNING: expected output {raw_out!r} not found.')
    print('Files matching cms_QA_test* in output dir:')
    for f in sorted(os.listdir(INFER_DIR)):
        if 'cms_QA_test' in f:
            print(f'  {f}')

print('Inference complete.')

## Cell 8 — Step 6: Evaluation (CPU, ~1 min)

In [ ]:
!python step6_full_evaluation.py \
    --ground_truth_csv  approach2_outputs/cms/cms_survey_distributions.csv \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --predictions_dir   approach2_outputs/cms \
    --weights_csv       approach2_outputs/cms/cms_subgroup_weights.csv \
    --output_dir        approach2_outputs/cms/evaluation

import os
for f in sorted(os.listdir('approach2_outputs/cms/evaluation')):
    print(' ', f)

## Cell 9 — Save all results to Drive

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/Subpop_replication_outputs'

# Copy everything in approach2_outputs/cms to Drive
src = 'approach2_outputs/cms'
dst = os.path.join(DRIVE_DIR, 'cms')
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print(f'Results saved to Google Drive: {dst}')
print('Files saved:')
for root, dirs, files in os.walk(dst):
    dirs[:] = [d for d in dirs if d != 'finetuning_data']  # skip large CSVs
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), dst)
        print(' ', rel)

---
## (Optional) Cell 10 — Run Mistral-7B for model comparison

Skip this if you only need Llama results.

Re-runs zero-shot + fine-tuning + inference with `mistralai/Mistral-7B-v0.1` and saves
results under `approach2_outputs/cms_mistral/` so the two model runs stay separate.

**Why bother?**  Comparing Llama vs Mistral shows whether the SubPop methodology is
model-agnostic — a meaningful finding for the paper.

In [ ]:
import os, glob, pathlib

MISTRAL    = 'mistralai/Mistral-7B-v0.1'
MISTRAL_OUT = 'approach2_outputs/cms_mistral'
os.makedirs(MISTRAL_OUT, exist_ok=True)

# ── Zero-shot baselines ──────────────────────────────────────────────────────
!python step2_vllm_baselines.py \
    --mode zeroshot \
    --model_name {MISTRAL} \
    --demographics_csv  approach2_outputs/cms/cms_demographics.csv \
    --steering_json     approach2_outputs/cms/cms_steering_prompts.json \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --output_dir        {MISTRAL_OUT} \
    --formats QA BIO PORTRAY

# ── LoRA fine-tuning ─────────────────────────────────────────────────────────
# Reuses the same finetuning_data/ splits generated in Cell 5.
os.chdir('subpop-main')
!python scripts/experiment/run_finetune.py \
    --model_name        {MISTRAL} \
    --dataset           cms_dataset \
    --steering_type     QA \
    --output_dir        ../{MISTRAL_OUT}/checkpoints_QA/ \
    --num_epochs        50 \
    --batch_size_training 4 \
    --lr                2e-4 \
    --use_peft \
    --quantization \
    --loss_function_type ce
os.chdir('..')

# ── Inference ────────────────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(f'{MISTRAL_OUT}/checkpoints_QA/*'))
if not checkpoints:
    raise FileNotFoundError(f'No Mistral checkpoint found under {MISTRAL_OUT}/checkpoints_QA/')
checkpoint_path = checkpoints[-1]
print(f'Using checkpoint: {checkpoint_path}')

TEST_CSV  = 'approach2_outputs/cms/finetuning_data/cms_QA_test.csv'
LORA_NAME = 'cms_mistral_QA'
INFER_DIR = f'{MISTRAL_OUT}/'

os.chdir('subpop-main')
!python scripts/experiment/run_inference.py \
    --input_paths             ../{TEST_CSV} \
    --output_dir              ../{INFER_DIR} \
    --base_model_name_or_path {MISTRAL} \
    --lora_path               ../{checkpoint_path} \
    --lora_name               {LORA_NAME}
os.chdir('..')

# Rename to canonical name
raw_stem  = pathlib.Path(TEST_CSV).stem
raw_out   = f'{INFER_DIR}{raw_stem}_{LORA_NAME}.csv'
canonical = f'{MISTRAL_OUT}/results_finetuned_QA.csv'
if os.path.exists(raw_out):
    os.rename(raw_out, canonical)
    print(f'Renamed: {raw_out} → {canonical}')
else:
    print(f'WARNING: {raw_out!r} not found.')

# ── Evaluation ───────────────────────────────────────────────────────────────
!python step6_full_evaluation.py \
    --ground_truth_csv  approach2_outputs/cms/cms_survey_distributions.csv \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --predictions_dir   {MISTRAL_OUT} \
    --weights_csv       approach2_outputs/cms/cms_subgroup_weights.csv \
    --output_dir        {MISTRAL_OUT}/evaluation

print('Mistral run complete. Results in', MISTRAL_OUT)